In [ ]:
import os
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
from imblearn.combine import SMOTETomek
from collections import Counter

print("Imports done!")

Imports done!


In [ ]:
os.makedirs('/content/data', exist_ok=True)

files = [
    'bio-decagon-combo.tar.gz',
    'bio-decagon-targets.tar.gz',
]
base_url = 'http://snap.stanford.edu/decagon'

for f in files:
    out = f'/content/data/{f}'
    if not os.path.exists(out.replace('.tar.gz', '.csv')):
        print(f"Downloading {f}...")
        os.system(f'wget -q {base_url}/{f} -O {out}')
        os.system(f'tar -xzf {out} -C /content/data/')
        print(f"Done: {f}")
    else:
        print(f"Already exists: {f}")

print("\nFiles in data folder:")
print([f for f in os.listdir('/content/data') if f.endswith('.csv')])


Done: bio-decagon-combo.tar.gz
Done: bio-decagon-targets.tar.gz

Files in data folder:
['._bio-decagon-combo.csv', 'bio-decagon-combo.csv', '._bio-decagon-targets.csv', 'bio-decagon-targets.csv']


In [ ]:
print("Loading real dataset...")
combo   = pd.read_csv('/content/data/bio-decagon-combo.csv')
targets = pd.read_csv('/content/data/bio-decagon-targets.csv')

drug1_col   = 'STITCH 1'
drug2_col   = 'STITCH 2'
se_col      = 'Polypharmacy Side Effect'
se_name_col = 'Side Effect Name'

se_counts      = combo[se_col].value_counts()
valid_se       = se_counts[se_counts >= 500].index
combo_filtered = combo[combo[se_col].isin(valid_se)].copy()

top25_common = se_counts[se_counts >= 500].head(25).index
top25_rare   = se_counts[se_counts >= 500].tail(25).index
mixed_se     = top25_common.tolist() + top25_rare.tolist()

combo_filtered = combo_filtered[
    combo_filtered[se_col].isin(mixed_se)
].copy()

common_se = top25_common.tolist()
rare_se   = top25_rare.tolist()

all_drugs = pd.unique(combo_filtered[[drug1_col, drug2_col]].values.ravel())
drug2idx  = {d: i for i, d in enumerate(all_drugs)}
n_drugs   = len(all_drugs)

print(f"Total side effects available: {len(valid_se)}")
print(f"Using mixed dataset: 25 common + 25 rare = 50 total")
print(f"Total triples: {len(combo_filtered):,}")
print(f"Drugs in subset: {n_drugs}")


Loading real dataset...
Total side effects available: 963
Using mixed dataset: 25 common + 25 rare = 50 total
Total triples: 566,602
Drugs in subset: 645


In [ ]:
prot_targets = targets.copy()
all_prots    = prot_targets.iloc[:, 1].unique()
prot2idx     = {p: i for i, p in enumerate(all_prots)}
n_prots      = len(all_prots)

rows, cols = [], []
for _, row in prot_targets.iterrows():
    drug = row.iloc[0]
    prot = row.iloc[1]
    if drug in drug2idx and prot in prot2idx:
        rows.append(drug2idx[drug])
        cols.append(prot2idx[prot])

drug_prot_adj   = sp.csr_matrix((np.ones(len(rows)), (rows, cols)),
                                 shape=(n_drugs, n_prots))
drug_prot_dense = drug_prot_adj.toarray()

pca           = PCA(n_components=50, random_state=42)
drug_features = pca.fit_transform(drug_prot_dense)
print(f"Drug features shape: {drug_features.shape}")
print(f"Variance explained:  {pca.explained_variance_ratio_.sum():.1%}")

def build_pair_features(d1_ids, d2_ids, feats):
    """150-dim: drug A (50) + drug B (50) + element-wise product (50)"""
    f1 = feats[d1_ids]
    f2 = feats[d2_ids]
    return np.hstack([f1, f2, f1 * f2])


Drug features shape: (645, 50)
Variance explained:  98.0%


In [ ]:
class DrugPairMLP(nn.Module):
    """
    MLP for drug-drug interaction prediction.
    Input:  150-dimensional drug pair feature vector
    Output: probability that this pair causes the side effect
    """
    def __init__(self, input_dim=150, hidden_dims=[256, 128, 64], dropout=0.3):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h in hidden_dims:
            layers += [
                nn.Linear(prev_dim, h),
                nn.BatchNorm1d(h),
                nn.ReLU(),
                nn.Dropout(dropout)
            ]
            prev_dim = h
        layers.append(nn.Linear(prev_dim, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return torch.sigmoid(self.network(x)).squeeze(-1)

print("DrugPairMLP defined!")


DrugPairMLP defined!


In [ ]:
def train_mlp(X_train, y_train, X_val, y_val,
              hidden_dims=[256, 128, 64],
              lr=1e-3, max_epochs=30, patience=5, batch_size=256):
    """
    Trains DrugPairMLP with early stopping based on validation AUPRC.
    """
    model     = DrugPairMLP(input_dim=X_train.shape[1], hidden_dims=hidden_dims)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.BCELoss()

    X_tr = torch.FloatTensor(X_train)
    y_tr = torch.FloatTensor(y_train)
    X_v  = torch.FloatTensor(X_val)

    best_auprc = 0.0
    best_state = None
    no_improve = 0
    history    = []

    for epoch in range(max_epochs):
        model.train()
        idx = torch.randperm(len(X_tr))
        epoch_losses = []

        for start in range(0, len(X_tr), batch_size):
            batch_idx = idx[start:start + batch_size]
            optimizer.zero_grad()
            preds = model(X_tr[batch_idx])
            loss  = criterion(preds, y_tr[batch_idx])
            loss.backward()
            optimizer.step()
            epoch_losses.append(loss.item())

        model.eval()
        with torch.no_grad():
            val_preds = model(X_v).numpy()
        val_auroc = roc_auc_score(y_val, val_preds)
        val_auprc = average_precision_score(y_val, val_preds)

        history.append({
            'epoch': epoch + 1,
            'train_loss': np.mean(epoch_losses),
            'val_auroc': val_auroc,
            'val_auprc': val_auprc
        })

        if val_auprc > best_auprc + 0.001:
            best_auprc = val_auprc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break

    model.load_state_dict(best_state)
    return model, history


In [ ]:
def run_mlp_for_se(se_code, se_name, combo_df, drug2idx, drug_features, n_drugs,
                    use_smote=True, seed=42):
    """
    Builds features for one side effect, optionally applies SMOTETomek
    (valid here — MLP operates on flat feature vectors), trains the MLP,
    and evaluates on a real (non-synthetic) held-out test set.
    """
    se_df  = combo_df[combo_df[se_col] == se_code]
    d1_ids = se_df[drug1_col].map(drug2idx).dropna().astype(int).values
    d2_ids = se_df[drug2_col].map(drug2idx).dropna().astype(int).values

    valid  = (d1_ids < n_drugs) & (d2_ids < n_drugs)
    d1_ids, d2_ids = d1_ids[valid], d2_ids[valid]

    if len(d1_ids) < 50:
        print(f"  Skipping {se_name} — too few pairs ({len(d1_ids)})")
        return None

    X_pos = build_pair_features(d1_ids, d2_ids, drug_features)
    y_pos = np.ones(len(X_pos))

    pos_set = set(zip(d1_ids.tolist(), d2_ids.tolist()))
    np.random.seed(seed)
    neg_pairs = []
    n_neg     = len(X_pos) * 3
    attempts  = 0
    while len(neg_pairs) < n_neg and attempts < n_neg * 20:
        r = np.random.randint(0, n_drugs)
        c = np.random.randint(0, n_drugs)
        if r != c and (r, c) not in pos_set and (c, r) not in pos_set:
            neg_pairs.append((r, c))
        attempts += 1

    neg_pairs = np.array(neg_pairs)
    X_neg = build_pair_features(neg_pairs[:, 0], neg_pairs[:, 1], drug_features)
    y_neg = np.zeros(len(X_neg))

    X = np.vstack([X_pos, X_neg])
    y = np.concatenate([y_pos, y_neg])

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed, stratify=y
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_train, y_train, test_size=0.1, random_state=seed, stratify=y_train
    )

    np.random.seed(seed + 999)
    pos_test_idx = np.where(y_test == 1)[0]
    neg_test_idx = np.where(y_test == 0)[0]
    sampled_neg  = np.random.choice(neg_test_idx, len(pos_test_idx), replace=False)
    keep         = np.concatenate([pos_test_idx, sampled_neg])
    X_test, y_test = X_test[keep], y_test[keep]

    scaler         = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled   = scaler.transform(X_val)
    X_test_scaled  = scaler.transform(X_test)

    if use_smote:
        try:
            smote = SMOTETomek(random_state=seed)
            X_train_final, y_train_final = smote.fit_resample(X_train_scaled, y_train)
        except Exception as e:
            print(f"  {se_name}: SMOTE failed ({e}), using original training data")
            X_train_final, y_train_final = X_train_scaled, y_train
    else:
        X_train_final, y_train_final = X_train_scaled, y_train

    model, history = train_mlp(
        X_train_final, y_train_final, X_val_scaled, y_val,
        max_epochs=30, patience=5
    )

    model.eval()
    with torch.no_grad():
        test_preds = model(torch.FloatTensor(X_test_scaled)).numpy()

    test_auroc = roc_auc_score(y_test, test_preds)
    test_auprc = average_precision_score(y_test, test_preds)

    return {
        'se_name':  se_name,
        'se_code':  se_code,
        'n_pairs':  len(d1_ids),
        'auroc':    test_auroc,
        'auprc':    test_auprc,
        'epochs_trained': len(history)
    }

print("run_mlp_for_se defined!")


run_mlp_for_se defined!


In [ ]:
mlp_results_common = []
mlp_results_rare    = []

print("\n── Common side effects (25) — WITH SMOTETomek ──")
for se_code in common_se:
    se_name = combo_filtered[combo_filtered[se_col] == se_code][se_name_col].iloc[0]
    result  = run_mlp_for_se(se_code, se_name, combo_filtered, drug2idx,
                              drug_features, n_drugs, use_smote=True)
    if result:
        mlp_results_common.append(result)
        print(f"  {se_name:<40} AUROC: {result['auroc']:.3f}  AUPRC: {result['auprc']:.3f}")

print("\n── Rare side effects (25) — WITH SMOTETomek ──")
for se_code in rare_se:
    se_name = combo_filtered[combo_filtered[se_col] == se_code][se_name_col].iloc[0]
    result  = run_mlp_for_se(se_code, se_name, combo_filtered, drug2idx,
                              drug_features, n_drugs, use_smote=True)
    if result:
        mlp_results_rare.append(result)
        print(f"  {se_name:<40} AUROC: {result['auroc']:.3f}  AUPRC: {result['auprc']:.3f}")

print("\nMLP + SMOTE Summary:")
print("=" * 50)
print(f"{'Category':<20} {'AUROC':>8} {'AUPRC':>8}")
print("-" * 50)
print(f"{'Common SEs (25)':<20} {np.mean([r['auroc'] for r in mlp_results_common]):>8.4f} {np.mean([r['auprc'] for r in mlp_results_common]):>8.4f}")
print(f"{'Rare SEs (25)':<20} {np.mean([r['auroc'] for r in mlp_results_rare]):>8.4f} {np.mean([r['auprc'] for r in mlp_results_rare]):>8.4f}")
print(f"{'Overall (50)':<20} {np.mean([r['auroc'] for r in mlp_results_common + mlp_results_rare]):>8.4f} {np.mean([r['auprc'] for r in mlp_results_common + mlp_results_rare]):>8.4f}")



── Common side effects (25) — WITH SMOTETomek ──
  arterial pressure NOS decreased          AUROC: 0.743  AUPRC: 0.729
  anaemia                                  AUROC: 0.739  AUPRC: 0.722
  Difficulty breathing                     AUROC: 0.746  AUPRC: 0.737
  nausea                                   AUROC: 0.749  AUPRC: 0.735
  neumonia                                 AUROC: 0.735  AUPRC: 0.720
  Fatigue                                  AUROC: 0.742  AUPRC: 0.724
  Pain                                     AUROC: 0.750  AUPRC: 0.739
  diarrhea                                 AUROC: 0.737  AUPRC: 0.722
  asthenia                                 AUROC: 0.752  AUPRC: 0.733
  emesis                                   AUROC: 0.744  AUPRC: 0.727
  edema extremities                        AUROC: 0.752  AUPRC: 0.734
  body temperature increased               AUROC: 0.713  AUPRC: 0.699
  pleural pain                             AUROC: 0.757  AUPRC: 0.742
  abdominal pain                        

In [ ]:
mlp_no_smote_common = []
mlp_no_smote_rare   = []

print("\n── Common side effects (25) — NO SMOTETomek (baseline) ──")
for se_code in common_se:
  se_name = combo_filtered[combo_filtered[se_col] == se_code][se_name_col].iloc[0]
  result  = run_mlp_for_se(se_code, se_name, combo_filtered, drug2idx,
                               drug_features, n_drugs, use_smote=False)
  if result:
    mlp_no_smote_common.append(result)
    print(f"  {se_name:<40} AUROC: {result['auroc']:.3f}  AUPRC: {result['auprc']:.3f}")

print("\n── Rare side effects (25) — NO SMOTETomek (baseline) ──")
for se_code in rare_se:
    se_name = combo_filtered[combo_filtered[se_col] == se_code][se_name_col].iloc[0]
    result  = run_mlp_for_se(se_code, se_name, combo_filtered, drug2idx,
                              drug_features, n_drugs, use_smote=False)
    if result:
        mlp_no_smote_rare.append(result)
        print(f"  {se_name:<40} AUROC: {result['auroc']:.3f}  AUPRC: {result['auprc']:.3f}")
print("\nMLP NO SMOTE Summary:")
print("=" * 50)
print(f"{'Category':<20} {'AUROC':>8} {'AUPRC':>8}")
print("-" * 50)
print(f"{'Common SEs (25)':<20} {np.mean([r['auroc'] for r in mlp_no_smote_common]):>8.4f} {np.mean([r['auprc'] for r in mlp_no_smote_common]):>8.4f}")
print(f"{'Rare SEs (25)':<20} {np.mean([r['auroc'] for r in mlp_no_smote_rare]):>8.4f} {np.mean([r['auprc'] for r in mlp_no_smote_rare]):>8.4f}")
print(f"{'Overall (50)':<20} {np.mean([r['auroc'] for r in mlp_no_smote_common + mlp_no_smote_rare]):>8.4f} {np.mean([r['auprc'] for r in mlp_no_smote_common + mlp_no_smote_rare]):>8.4f}")


── Common side effects (25) — NO SMOTETomek (baseline) ──
  arterial pressure NOS decreased          AUROC: 0.743  AUPRC: 0.727
  anaemia                                  AUROC: 0.743  AUPRC: 0.725
  Difficulty breathing                     AUROC: 0.747  AUPRC: 0.735
  nausea                                   AUROC: 0.747  AUPRC: 0.734
  neumonia                                 AUROC: 0.736  AUPRC: 0.718
  Fatigue                                  AUROC: 0.749  AUPRC: 0.734
  Pain                                     AUROC: 0.747  AUPRC: 0.738
  diarrhea                                 AUROC: 0.742  AUPRC: 0.725
  asthenia                                 AUROC: 0.751  AUPRC: 0.733
  emesis                                   AUROC: 0.745  AUPRC: 0.728
  edema extremities                        AUROC: 0.751  AUPRC: 0.736
  body temperature increased               AUROC: 0.714  AUPRC: 0.699
  pleural pain                             AUROC: 0.753  AUPRC: 0.739
  abdominal pain               